# Mongo tutorial

## Prerequisites

### Documentation

You will find all documentation for :
* [Mongo commands](https://docs.mongodb.com/manual/reference/)
* [Mongo python client](http://api.mongodb.com/python/current/api/pymongo/mongo_client.html#pymongo.mongo_client.MongoClient)

### Import libraries

In [1]:
import datetime
from pprint import pprint

import pymongo
from pymongo import MongoClient

In [2]:
client = MongoClient('localhost', 27017)

In [3]:
# let's work in a test_database
db = client.test_database
posts = db.posts

In [4]:
post = {
    "author": "Mike",
    "text": "My first blog post!",
    "tags": ["mongodb", "python", "pymongo"],
    "date": datetime.datetime.utcnow()
}
post_id = posts.insert_one(post).inserted_id
post_id

ObjectId('6996d349390a58762cf5b2fd')

In [5]:
db.list_collection_names()

['posts']

In [6]:
pprint(posts.find_one())

{'_id': ObjectId('6996d349390a58762cf5b2fd'),
 'author': 'Mike',
 'date': datetime.datetime(2026, 2, 19, 9, 9, 29, 20000),
 'tags': ['mongodb', 'python', 'pymongo'],
 'text': 'My first blog post!'}


You can launch a terminal aside, connect to your server with a mongo client and check that the value is present :

```bash
vagrant@nosql:~$ mongo
> show databases;
admin          0.000GB
config         0.000GB
local          0.000GB
test_database  0.000GB
> use test_database;
switched to db test_database
> db.posts.find()
{ 
    "_id" : ObjectId("..."), 
    "author" : "Mike", 
    "text" : "My first blog post!", 
    "tags" : [ "mongodb", "python", "pymongo"], 
    "date" : ISODate("2019-02-10T11:33:47.883Z") 
}
```

## I. Quick start

### First steps

**Q** : Create a document `{msg: 'hello'}` in the `test` collection with `insert_one()`. Fetch it back to display it. What is the `_id` for ?

NB : if the collection doesn't exist yet, MongoDB automatically creates it.

In [13]:
#Collection test 
test_collection = db.test
#Nouveau document
new_doc = {'msg': 'hello'}
test_collection.insert_one(new_doc)

#Récupérer le document
fetched_msg = test_collection.find_one({'msg': 'hello'})

pprint(fetched_msg)

{'_id': ObjectId('6996d5a4390a58762cf5b2fe'), 'msg': 'hello'}


**Q**: Display the number of documents inside the `test` collection

In [14]:
count = test_collection.count_documents({})

print({count})

{3}


### Interacting with a database

We have 2 `.json` files we want to interact with inside the `data` folder. Let's first dump them into a `MovieLens` database, inside `users` and `movies` collections.

For this section, you will need to read a bit on [query operators](https://docs.mongodb.com/manual/reference/operator/query/#query-selectors). Most methods on collections you will use have `filter` as a first parameter, on which you must pass a dictionary of query parameters.

**Q** : In the `MovieLens` database, load `data/movielens_movies.json` into `movies` and `data/movielens_users.json` into `users`. 

Use the dedicated shell command for this : `mongoimport --db <some_db> --collection <some_collection> --file <some_file>` 

In [15]:
# Importation des films dans la collection 'movies'
mongoimport --db MovieLens --collection movies --file data/movielens_movies.json

# Importation des utilisateurs dans la collection 'users'
mongoimport --db MovieLens --collection users --file data/movielens_users.json

SyntaxError: invalid syntax (1913603113.py, line 2)

**Q** : how many users are in the `MovieLens` database ?

In [17]:
users_col = client.MovieLens.users
user_count=users_col.count_documents({})
print(user_count)

6040


**Q** : Display all comedies (the `genres` property equals `Comedy`). 

NB : You will need to find how to go through a `command_cursor`, then use the `pprint` function for a better display of those documents.

In [20]:
#query selector
query = {"genres": "Comedy"}

comedies_cursor = client.MovieLens.movies.find(query)

for movie in comedies_cursor:
    pprint(movie)

{'_id': 5, 'genres': 'Comedy', 'title': 'Father of the Bride Part II (1995)'}
{'_id': 19,
 'genres': 'Comedy',
 'title': 'Ace Ventura: When Nature Calls (1995)'}
{'_id': 38, 'genres': 'Comedy', 'title': 'It Takes Two (1995)'}
{'_id': 52, 'genres': 'Comedy', 'title': 'Mighty Aphrodite (1995)'}
{'_id': 63,
 'genres': 'Comedy',
 'title': "Don't Be a Menace to South Central While Drinking Your Juice in the "
          'Hood (1996)'}
{'_id': 69, 'genres': 'Comedy', 'title': 'Friday (1995)'}
{'_id': 65, 'genres': 'Comedy', 'title': 'Bio-Dome (1996)'}
{'_id': 88, 'genres': 'Comedy', 'title': 'Black Sheep (1996)'}
{'_id': 96, 'genres': 'Comedy', 'title': 'In the Bleak Midwinter (1995)'}
{'_id': 101, 'genres': 'Comedy', 'title': 'Bottle Rocket (1996)'}
{'_id': 102, 'genres': 'Comedy', 'title': 'Mr. Wrong (1996)'}
{'_id': 104, 'genres': 'Comedy', 'title': 'Happy Gilmore (1996)'}
{'_id': 109, 'genres': 'Comedy', 'title': 'Headless Body in Topless Bar (1995)'}
{'_id': 115, 'genres': 'Comedy', 'tit

**Q** : Fetch and display the `name` and `occupation` for Clifford Johnathan. The second paramater for `find()` ([doc here](https://api.mongodb.com/python/current/api/pymongo/collection.html#pymongo.collection.Collection.find)) is called the `projection` and is used to limit which data to fetch from the query.

In [21]:
#filtre de recherche
query = {"name": "Clifford Johnathan"}

#projection
projection = {"name": 1, "occupation": 1, "_id": 0}

#affichage 
print(client.MovieLens.users.find_one(query, projection))

{'name': 'Clifford Johnathan', 'occupation': 'technician/engineer'}


**Q**: How many minors (by `age`) have rated movies ?

In [22]:
#filtre
query_minors = {"age": {"$lt": 18}}

minors_count = client.MovieLens.users.count_documents(query_minors)
print(minors_count)

222


**Q**: Display science fiction movies ('Sci-Fi') and suspense movies ('Thriller'). This time you need to use a regex to parse genres and look for those values.

In [23]:
genre_regex = "Sci-Fi|Thriller"
#définir filtre
query = {"genres": {"$regex": genre_regex, "$options": "i"}}

movies_cursor = client.MovieLens.movies.find(query)

for movie in movies_cursor:
    
    print(f"Titre : {movie.get('title')} | Genres : {movie.get('genres')}")

Titre : Heat (1995) | Genres : Action|Crime|Thriller
Titre : GoldenEye (1995) | Genres : Action|Adventure|Thriller
Titre : Four Rooms (1995) | Genres : Thriller
Titre : Casino (1995) | Genres : Drama|Thriller
Titre : Copycat (1995) | Genres : Crime|Drama|Thriller
Titre : Assassins (1995) | Genres : Thriller
Titre : Powder (1995) | Genres : Drama|Sci-Fi
Titre : City of Lost Children, The (1995) | Genres : Adventure|Sci-Fi
Titre : Twelve Monkeys (1995) | Genres : Drama|Sci-Fi
Titre : Seven (Se7en) (1995) | Genres : Crime|Thriller
Titre : Usual Suspects, The (1995) | Genres : Crime|Thriller
Titre : Guardian Angel (1994) | Genres : Action|Drama|Thriller
Titre : Eye for an Eye (1996) | Genres : Drama|Thriller
Titre : Lawnmower Man 2: Beyond Cyberspace (1996) | Genres : Sci-Fi|Thriller
Titre : From Dusk Till Dawn (1996) | Genres : Action|Comedy|Crime|Horror|Thriller
Titre : Juror, The (1996) | Genres : Drama|Thriller
Titre : Screamers (1995) | Genres : Sci-Fi|Thriller
Titre : Mary Reilly (19

**Q**: If we want more advanced textual search, we need a particular index. Use the `create_index()` method to index as [TEXT](https://docs.mongodb.com/manual/core/index-text/) the `genres` field of the `movies` collection.

In [24]:
movies_collection = client.MovieLens.movies

index_name = movies_collection.create_index([("genres", pymongo.TEXT)])



genres_text


**Q**: Restart the search for science fiction and thriller movies with the operator `$text`

In [25]:
#définir filtre
text_query = {"$text": {"$search": "Sci-Fi Thriller"}}

movies_cursor = client.MovieLens.movies.find(text_query)
for movie in movies_cursor:
    print(f"Titre : {movie.get('title')} | Genres : {movie.get('genres')}")

Titre : Kronos (1957) | Genres : Sci-Fi
Titre : X: The Unknown (1956) | Genres : Sci-Fi
Titre : Rocketship X-M (1950) | Genres : Sci-Fi
Titre : Project Moon Base (1953) | Genres : Sci-Fi
Titre : Light Years (1988) | Genres : Sci-Fi
Titre : Quatermass and the Pit (1967) | Genres : Sci-Fi
Titre : Devil Girl From Mars (1954) | Genres : Sci-Fi
Titre : Destination Moon (1950) | Genres : Sci-Fi
Titre : Mission to Mars (2000) | Genres : Sci-Fi
Titre : Omega Man, The (1971) | Genres : Sci-Fi
Titre : Zone 39 (1997) | Genres : Sci-Fi
Titre : Thing From Another World, The (1951) | Genres : Sci-Fi
Titre : It Conquered the World (1956) | Genres : Sci-Fi
Titre : Mole People, The (1956) | Genres : Sci-Fi
Titre : Earth Vs. the Flying Saucers (1956) | Genres : Sci-Fi
Titre : It Came from Beneath the Sea (1955) | Genres : Sci-Fi
Titre : It Came from Outer Space (1953) | Genres : Sci-Fi
Titre : Flying Saucer, The (1950) | Genres : Sci-Fi
Titre : Sticky Fingers of Time, The (1997) | Genres : Sci-Fi
Titre 

**Q**: Display the first 30 movies (`limit`) in alphabetical order (`sort`) by title

In [26]:
#critère de tri
alphabetical_sort = [("title", 1)]

movies_cursor = client.MovieLens.movies.find({}).sort(alphabetical_sort).limit(30)
for movie in movies_cursor:
    print(f"- {movie.get('title')}")

- $1,000,000 Duck (1971)
- 'Night Mother (1986)
- 'Til There Was You (1997)
- 'burbs, The (1989)
- ...And Justice for All (1979)
- 1-900 (1994)
- 10 Things I Hate About You (1999)
- 101 Dalmatians (1961)
- 101 Dalmatians (1996)
- 12 Angry Men (1957)
- 13th Warrior, The (1999)
- 187 (1997)
- 2 Days in the Valley (1996)
- 20 Dates (1998)
- 20,000 Leagues Under the Sea (1954)
- 200 Cigarettes (1999)
- 2001: A Space Odyssey (1968)
- 2010 (1984)
- 24 7: Twenty Four Seven (1997)
- 24-hour Woman (1998)
- 28 Days (2000)
- 3 Ninjas: High Noon On Mega Mountain (1998)
- 3 Strikes (2000)
- 301, 302 (1995)
- 39 Steps, The (1935)
- 400 Blows, The (Les Quatre cents coups) (1959)
- 42 Up (1998)
- 52 Pick-Up (1986)
- 54 (1998)
- 7th Voyage of Sinbad, The (1958)


**Q**: How many users have seen the movie "Star Wars: Episode V - The Empire Strikes Back (1980)" (`_id 1196`) ? The `movies` argument is an array so we should try the [elemMatch](https://docs.mongodb.com/manual/reference/operator/projection/elemMatch/) operator here.

In [37]:
# vérification des clés
users_collection = client.MovieLens.users

first_user = users_collection.find({}).limit(1)

# 3. Parcourir le curseur et afficher le contenu complet avec pprint
for user in first_user:
    pprint(user)
    print("-" * 20) 

{'_id': 6038,
 'age': 95,
 'gender': 'F',
 'movies': [{'movieid': 1419, 'rating': 4, 'timestamp': 956714815},
            {'movieid': 920, 'rating': 3, 'timestamp': 956706827},
            {'movieid': 3088, 'rating': 5, 'timestamp': 956707640},
            {'movieid': 232, 'rating': 4, 'timestamp': 956707640},
            {'movieid': 1136, 'rating': 4, 'timestamp': 956707708},
            {'movieid': 1148, 'rating': 5, 'timestamp': 956707604},
            {'movieid': 1183, 'rating': 5, 'timestamp': 956717204},
            {'movieid': 2146, 'rating': 4, 'timestamp': 956706909},
            {'movieid': 3548, 'rating': 4, 'timestamp': 956707604},
            {'movieid': 356, 'rating': 4, 'timestamp': 956707005},
            {'movieid': 1210, 'rating': 4, 'timestamp': 956706876},
            {'movieid': 1223, 'rating': 5, 'timestamp': 956707734},
            {'movieid': 1276, 'rating': 3, 'timestamp': 956707604},
            {'movieid': 1296, 'rating': 5, 'timestamp': 956714684},
         

In [33]:
#définir filtre
query = {"movies": {"$elemMatch": {"movieid": 1196}}}

viewers_count = client.MovieLens.users.count_documents(query)

print(viewers_count)

2990


**Q**: And how many gave it a rating of 1 or 2 ?

In [38]:
#définir filtre

query = {
    "movies": {
        "$elemMatch": {
            "movieid": 1196, 
            "rating": {"$in": [1, 2]}
        }
    }
}

users_count = client.MovieLens.users.count_documents(query)

print(users_count)

105


### Updating data

**Q**: Insert a new user with the properties `name`, `gender` ('M' or'F'), `occupation` and `age`, using the `insert_one()` command. Display it with `find_one()`.

In [40]:
#nouvel utilisateur 
new_user = {
    "name": "Anne-Camille Vial",
    "gender": "F",
    "occupation": "consultant",
    "age": 36
}

insert_result = client.MovieLens.users.insert_one(new_user)

fetched_user = client.MovieLens.users.find_one({"name": "Anne-Camille Vial"})
pprint(fetched_user)

{'_id': ObjectId('6996e852390a58762cf5b301'),
 'age': 36,
 'gender': 'F',
 'name': 'Anne-Camille Vial',
 'occupation': 'consultant'}


**Q**: Add an appreciation on a viewed movie with `update_one()`, add the movies property containing a table with a document (`movieid`, `rating`, `timestamp` with the value `datetime.datetime.utcnow()`).

You will need to read the documentation on [update operators](https://docs.mongodb.org/manual/reference/operator/update/).

In [41]:

user_filter = {"name": "Anne-Camille Vial"}


new_movie_rating = {
    "movieid": 1196, 
    "rating": 3,
    "timestamp": datetime.datetime.utcnow()
}


client.MovieLens.users.update_one(
    user_filter, 
    {"$push": {"movies": new_movie_rating}}
)


updated_user = client.MovieLens.users.find_one(user_filter)

pprint(updated_user)

{'_id': ObjectId('6996e852390a58762cf5b301'),
 'age': 36,
 'gender': 'F',
 'movies': [{'movieid': 1196,
             'rating': 3,
             'timestamp': datetime.datetime(2026, 2, 19, 10, 45, 41, 93000)}],
 'name': 'Anne-Camille Vial',
 'occupation': 'consultant'}


**Q**: Find the number of users who have declared a `programmer` occupation. Modify them so that they are `developer`. Verify your update.

In [44]:
programmer_query = {"occupation": "programmer"}
count_programmer = client.MovieLens.users.count_documents(programmer_query)
print(f"Nombre de 'programmer' avant mise à jour : {count_programmer}")

# Modifier tous les "programmer" en "developer"

update_result = client.MovieLens.users.update_many(
    programmer_query,
    {"$set": {"occupation": "developer"}}
)
print(f"Nombre de documents modifiés : {update_result.modified_count}")

# Vérifier la mise à jour

count_after_dev = client.MovieLens.users.count_documents({"occupation": "developer"})
count_after_prog = client.MovieLens.users.count_documents({"occupation": "programmer"})

print(f"\nVérification :")
print(f"- Nouveaux 'developer' : {count_after_dev}")
print(f"- Anciens 'programmer' restants : {count_after_prog}")

Nombre de 'programmer' avant mise à jour : 388
Nombre de documents modifiés : 388

Vérification :
- Nouveaux 'developer' : 388
- Anciens 'programmer' restants : 0


## II. Modelling a blog

We will now model a blog using Mongo. 

First, switch to a new `Blog` database. Each blog post will have the following arguments:

* The author (author field, string type)
* The date (date field, string type in YYYY-MM-DD format)
* The content (field content)
* Tags (field tags, a string array)
* A list of comments (field comments) containing:
 * The author (author field, string type)
 * The date (date field, string type in YYYY-MM-DD format)
 * The content (field content)


**Q**: Create a first post by `rick`, on January 15th, with the tags `mongodb` and `nosql`.

In [45]:
#nouvelle base de données
blog_db = client.Blog
posts_collection = blog_db.posts

#nouveau post
first_post = {
    "author": "rick",
    "date": "2026-01-15",
    "content": "Ceci est mon premier article sur MongoDB et NoSQL !",
    "tags": ["mongodb", "nosql"],
    "comments": [] 
}

insert_result = posts_collection.insert_one(first_post)

**Q**: Create a second post by `kate`, on January 21, with the tag `nosql` and a comment from `rick` on the same day.

In [52]:
comment_content = {
    "author": "rick",
    "date": "2026-01-21",
    "content": "Je crois que j'avais déjà parlé de ces éléments la semaine dernière..."
}
second_post = {
    "author": "kate",
    "date": "2026-01-21",
    "content": "Ceci est mon premier article sur NoSQL",
    "tags": ["nosql"],
    "comments": [] 
}

posts_collection.insert_one(second_post)
posts_collection.update_one(
    {"author": "kate", "date": "2026-01-21"},
    {"$set": {"comments": [comment_content]}}
)

pprint(posts_collection.find_one({"author": "kate"}))

{'_id': ObjectId('6996ed5b390a58762cf5b304'),
 'author': 'kate',
 'comments': [{'author': 'rick',
               'content': "Je crois que j'avais déjà parlé de ces éléments la "
                          'semaine dernière...',
               'date': '2026-01-21'}],
 'content': 'Ceci est mon premier article sur NoSQL',
 'date': '2026-01-21',
 'tags': ['nosql']}


**Q**: Display the author of the last post with the tag `nosql`

In [47]:
# définr filtre
query = {"tags": "nosql"}

# Trier par date décroissante (-1) pour avoir le "dernier" post en premier

last_post = posts_collection.find_one(
    query, 
    sort=[("date", -1)]
)

# Afficher l'auteur

print(f"L'auteur du dernier post avec le tag 'nosql' est : {last_post['author']}")


L'auteur du dernier post avec le tag 'nosql' est : kate


**Q**: Add a comment by `jack` on January 25, to `kate`'s post

In [53]:
comment_info = {
    "author": "jack",
    "date": "2026-01-25",
    "content": "Les éléments évoqués sont plus clairs que dans le post précédent"
}

posts_collection.update_one(
    {"author": "kate", "date": "2026-01-21"},
    {"$push": {"comments": comment_info}}
)

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

**Q**: Display all comments by `kate`

In [54]:
pipeline = [
    # ne traiter que les commentaires
    {"$unwind": "$comments"},
    
    # Filtre auteur="kate"
    {"$match": {"comments.author": "kate"}},
    
    # Projeter uniquement le contenu du commentaire
    {"$project": {
        "_id": 0,
        "post_author": "$author",
        "comment": "$comments"
    }}
]

# Exécuter l'agrégation
results = posts_collection.aggregate(pipeline)


print("--- Commentaires écrits par Kate ---")
for doc in results:
    pprint(doc)

--- Commentaires écrits par Kate ---


## Postquisites

In [ ]:
!mongo test_database --eval 'db.dropDatabase()'

In [ ]:
!mongo MovieLens --eval 'db.dropDatabase()'

In [ ]:
!mongo Blog --eval 'db.dropDatabase()'